# Overture MK2 Closure Sizing
Edited 10/16/2026, Joris Lindinger

Based on OVMK1 master document and Closure V2 technical documentation

Closure layout:
- Mounted to IRIS with 3 evenly distributed U-brackets on the injector-side flange
- Mounted to injector with bolts (TBD)
- Nozzle held in by a bolted thrust ring with 16 radial bolts (replaces the snap ring)

## General data

Thin-walled assumption valid, since diameter/thickness = 32 > 20. Thick-walled formulas are still used below since they are exact

Closure loads (injector side, nozzle end) are multiplied by the load multiplier K_d to account for the dynamic startup shock, the rest of the analysis is static

In [18]:
import math
import numpy as np

# geometry data
d_inner = 4 #inches, matched to nozzle outer diameter from DXF
t_wall = 0.125 #inches
d_outer = d_inner + 2*t_wall #inches
l = 12 #inches

# material data (304 stainless) STANDARD TEMPERATURE FOR NOW, CHECK AT HIGH T AGAIN!!!
sigma_yield = 29732.7 #yield strength, psi
tau_yield = 0.577*sigma_yield #shear yield strength based on van mises, psi
nu = 0.29 #poisson's ratio

# other data
p_chamber = 300 #psi
F_thrust = 1100 #lbf (4.9 kN)
K_d = 2.0 #load multiplier for dynamic startup shock
n_safety = 2 #safety factor 2, from OVMK1 documentation

## Hoop stress

Thick-walled formula

r =^ radius at point of interest<br>
A, B =^ boundary conditions

P_a =^ internal pressure<br>
P_b =^ external pressure<br>
a =^ inner radius<br>
b =^ outer radius<br>

In [19]:
P_a = p_chamber
P_b = 14.7 #psi, sea level assumed for now

a = d_inner/2
b = a + t_wall

r = a #using inner radius for worst-case estimate

A = (P_a*a**2 - P_b*b**2)/(b**2-a**2)
B = a**2*b**2*(P_a-P_b)/(b**2 - a**2)


sigma_hoop = A + B / r**2

print(f"Hoop stress: {sigma_hoop} psi")

Hoop stress: 4697.072727272727 psi


# Axial stress

This was somehow not included in the OVMK1 master document

Is uniformly distributed across the wall, so formula is simple

In [20]:
sigma_axial = (P_a*a**2-P_b*b**2)/(b**2-a**2)
print(f"Axial stress: {sigma_axial} psi")

Axial stress: 2198.5363636363636 psi


## Radial stress

A, B, r same as for hoop stress

In [21]:
sigma_rad = A - B / r**2

print(f"Radial stress: {sigma_rad} psi")

Radial stress: -300.0 psi


# Von Mises Stress

Source: (https://en.wikipedia.org/wiki/Von_Mises_yield_criterion), State of stress: Principal stresses

In [22]:
sigma_van_mises = math.sqrt(0.5*((sigma_hoop-sigma_axial)**2+(sigma_axial-sigma_rad)**2+(sigma_rad-sigma_hoop)**2))

print(f"Applied van mises stress: {sigma_van_mises} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_van_mises}")
if sigma_yield/sigma_van_mises < n_safety:
    print("FAIL!")

Applied van mises stress: 4327.591926376569 psi
Yield strength: 29732.7 psi
Safety factor: 6.870495302197951


# Closure loads

Chamber pressure acts on the full bore area and pushes the injector and the nozzle out of the chamber

$F_{eject} = P_c \frac{\pi D_{bore}^2}{4}$<br>
$F_{dyn} = K_d F_{eject}$

In [23]:
F_eject = p_chamber*math.pi*d_inner**2/4 #lbf
F_dyn = F_eject*K_d #lbf

print(f"Ejection force: {F_eject} lbf")
print(f"Dynamic ejection force: {F_dyn} lbf")

Ejection force: 3769.9111843077517 lbf
Dynamic ejection force: 7539.822368615503 lbf


# Injector side

Flange welded to the chamber, mounted to IRIS with 3 U-brackets

Assumptions:
- Materials stay in the elastic region
- Chamber is modeled as a rigid wall, so the flange acts like a cantilevered plate
- Bracket load disperses into the flange at 45°
- Bolt stress dissipates radially
- Weld is modeled as a fillet with an isosceles right triangle cross section

## Flange bending

Each bracket carries a third of the dynamic thrust at the bracket hole radius, cantilevered from the chamber

r_bracket =^ radius of the bracket holes<br>
L =^ lever arm, measured from the bore radius (conservative)<br>
w_bracket =^ bracket width<br>
b_eff =^ effective flange width at 45° dispersion<br>
b_max =^ maximum flange width per bracket (bore circumference / 3)<br>
t_flange =^ flange thickness

$\sigma = \frac{Mc}{I} = \frac{6M}{b t_{flange}^2}$, with $b = \min(b_{eff}, b_{max})$

In [24]:
r_bracket = 3.131 #inches
w_bracket = 2.5 #inches
t_flange = 5/16 #inches
n_brackets = 3

F_bracket = F_thrust*K_d/n_brackets #lbf
L = r_bracket - d_inner/2 #inches
M_flange = F_bracket*L #in-lbf

b_eff = w_bracket + 2*L*math.tan(math.radians(45)) #inches
b_max = math.pi*d_inner/n_brackets #inches
b_flange = min(b_eff, b_max) #inches

sigma_flange = 6*M_flange/(b_flange*t_flange**2)
t_flange_min = math.sqrt(6*M_flange*n_safety/(sigma_yield*b_flange)) #inches

print(f"Flange width used: {b_flange} in")
print(f"Applied bending stress: {sigma_flange} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_flange}")
print(f"Minimum flange thickness: {t_flange_min} in")
if sigma_yield/sigma_flange < n_safety:
    print("FAIL!")

Flange width used: 4.1887902047863905 in
Applied bending stress: 12165.406599206523 psi
Yield strength: 29732.7 psi
Safety factor: 2.4440366836517646
Minimum flange thickness: 0.28269046449752533 in


## Weld shear

Fillet weld around the chamber outer diameter carries the dynamic ejection force on the injector

t_weld =^ fillet weld leg size<br>
t_throat =^ weld throat thickness<br>
A_weld =^ weld throat area

$t_{throat} = t_{weld} \cos 45° = \frac{t_{weld}}{\sqrt{2}}$<br>
$\tau_{weld} = \frac{F_{dyn}}{A_{weld}} = \frac{\sqrt{2} F_{dyn}}{\pi D_{o} t_{weld}}$

In [25]:
t_weld = 1/8 #inches

t_throat = t_weld*math.cos(math.radians(45)) #inches
A_weld = math.pi*d_outer*t_throat #in^2
tau_weld = F_dyn/A_weld
t_weld_min = math.sqrt(2)*F_dyn*n_safety/(math.pi*d_outer*tau_yield) #inches

print(f"Applied shear stress: {tau_weld} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_weld}")
print(f"Minimum weld leg size: {t_weld_min} in")
if tau_yield/tau_weld < n_safety:
    print("FAIL!")

Applied shear stress: 6388.917740603158 psi
Yield shear strength: 17155.7679 psi
Safety factor: 2.6852385015025058
Minimum weld leg size: 0.09310159967545316 in


# Nozzle end

Bolted thrust ring (replaces the snap ring). The ring sits in the chamber bore behind the nozzle and is held by radial bolts through the chamber wall

Assumptions:
- Radial symmetry
- Linear elasticity: with all safety factors >= 2, everything stays below 50% of yield
- Bolts are in pure transverse shear: tight fit between ring and chamber, so no lever arm and negligible bolt bending
- Load is shared evenly across all bolts
- Saint-Venant's principle: stress concentrations at the drilled hole edges dissipate radially
- The crush seal deforms plastically and absorbs the thermal expansion of the thrust ring
- Thrust ring is 304 stainless, same as the chamber

n_bolts =^ number of radial bolts<br>
d_bolt =^ bolt major diameter<br>
n_threads =^ threads per inch<br>
e =^ edge distance from hole center to end of chamber<br>
t_p =^ thrust ring thickness (axial)<br>
r_i =^ thrust ring inner radius, slightly bigger than the nozzle exit radius (1.679 in)<br>
r_o =^ thrust ring outer radius (bore radius)

In [26]:
n_bolts = 16
d_bolt = 0.25 #inches
n_threads = 20 #threads per inch, 1/4-20 UNC assumed
e = 0.25 #inches
t_p = 0.5 #inches
r_i = 1.75 #inches
r_o = d_inner/2 #inches

# bolt material: 18-8 stainless, ASTM F593 condition CW (cold worked), 1/4 to 5/8 in
# CHECK AGAINST THE ACTUAL BOLTS!!! annealed 18-8 is only ~30 ksi yield, which fails the bolt shear check
sigma_yield_bolt = 65000 #yield strength, psi
tau_yield_bolt = 0.577*sigma_yield_bolt #shear yield strength based on van mises, psi

## Radial bolt shear

Shear plane at the interface between chamber wall and thrust ring. The ring is tapped right at the interface, so the threads cross the shear plane and the minor diameter area is used

$d_{minor} = d_{bolt} - \frac{1.299038}{n_{threads}}$ (Shigley Table 8-2)<br>
$\tau_{bolt} = \frac{F_{dyn}}{n \frac{\pi d_{minor}^2}{4}}$

In [27]:
d_minor = d_bolt - 1.299038/n_threads #inches, minor diameter (Shigley Table 8-2)
A_bolt = math.pi*d_minor**2/4 #in^2, minor diameter area
A_bolt_tot = n_bolts*A_bolt #in^2

tau_bolt = F_dyn/A_bolt_tot

print(f"Applied shear stress: {tau_bolt} psi")
print(f"Yield shear strength: {tau_yield_bolt} psi")
print(f"Safety factor: {tau_yield_bolt/tau_bolt}")
if tau_yield_bolt/tau_bolt < n_safety:
    print("FAIL!")

Applied shear stress: 17521.931968965124 psi
Yield shear strength: 37505.0 psi
Safety factor: 2.1404603137615714


## Chamber bearing

Bolts crush the holes in the chamber wall into ovals

$\sigma_{br,chamber} = \frac{F_{dyn}}{n d_{bolt} t_{wall}}$

In [28]:
A_br_chamber = n_bolts*d_bolt*t_wall #in^2
sigma_br_chamber = F_dyn/A_br_chamber

print(f"Applied bearing stress: {sigma_br_chamber} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_br_chamber}")
if sigma_yield/sigma_br_chamber < n_safety:
    print("FAIL!")

Applied bearing stress: 15079.644737231007 psi
Yield strength: 29732.7 psi
Safety factor: 1.971710906861832
FAIL!


## Thrust ring bearing

Bolts crush the holes in the thrust ring into ovals. Since the bolts are radial, the bearing length is the thread engagement in the ring, assumed to be the full radial width of the ring

l_engage =^ thread engagement length in the ring

$\sigma_{br,ring} = \frac{F_{dyn}}{n d_{bolt} l_{engage}}$

In [29]:
l_engage = r_o - r_i #inches
A_br_ring = n_bolts*d_bolt*l_engage #in^2
sigma_br_ring = F_dyn/A_br_ring

print(f"Applied bearing stress: {sigma_br_ring} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_br_ring}")
if sigma_yield/sigma_br_ring < n_safety:
    print("FAIL!")

Applied bearing stress: 7539.822368615503 psi
Yield strength: 29732.7 psi
Safety factor: 3.943421813723664


## Chamber edge tear-out

Bolts rip through the lip between the holes and the end of the chamber, two shear planes per bolt

$\tau_{tear} = \frac{F_{dyn}}{2 n e t_{wall}}$

In [30]:
A_tear = 2*n_bolts*e*t_wall #in^2
tau_tear = F_dyn/A_tear

print(f"Applied shear stress: {tau_tear} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_tear}")
if tau_yield/tau_tear < n_safety:
    print("FAIL!")

Applied shear stress: 7539.822368615503 psi
Yield shear strength: 17155.7679 psi
Safety factor: 2.275354386518554


## Thrust ring bending

Ring modeled as an annular plate, outer edge simply supported by the bolts, inner edge free, with the full dynamic load as a line load along the inner edge (worst case: nozzle only touches the inner lip of the ring). Max stress is the tangential stress at the inner edge

Note: The notion V2 document used Roark's formula for a *solid* circular plate with an annular line load, which ignores the hole and gives ~8.5x lower stress. ALSO THIS FORMULA IS FOR FLAT PLATES, SO WE SHOULD CHECK IF THERES A MORE ACCURATE ONE SINCE THIS GREATLY OVERESTIMATES THE STRESS (SO MUCH THAT IT CURRENTLY FAILS).

Source: Timoshenko & Woinowsky-Krieger, Theory of Plates and Shells, Ch. 3, circular plate with a circular hole at the center (Roark Table 11.2 case 1a with the load at the inner edge)

$\sigma_{max} = \frac{3F_{dyn}}{2\pi t_p^2}\left(\frac{2r_o^2(1+\nu)}{r_o^2-r_i^2}\ln\frac{r_o}{r_i} + (1-\nu)\right)$



In [34]:
sigma_ring = 3*F_dyn/(2*math.pi*t_p**2)*(2*r_o**2*(1+nu)/(r_o**2-r_i**2)*math.log(r_o/r_i) + (1-nu))
t_p_min = t_p*math.sqrt(n_safety*sigma_ring/sigma_yield) #inches, stress scales with 1/t_p^2

print(f"Applied bending stress: {sigma_ring} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_ring}")
print(f"Minimum ring thickness: {t_p_min} in")
if sigma_yield/sigma_ring < n_safety:
    print("FAIL!")

Applied bending stress: 31390.75540815472 psi
Yield strength: 29732.7 psi
Safety factor: 0.9471801367442089
Minimum ring thickness: 0.7265553603027077 in
FAIL!


# Temperature Dependence

Not used for now. Also, ill have to check which datasheet the values come from again

In [32]:
def yield_strength(T):
    """0.2% yield strength in psi at temperature T in deg C, linearly interpolated from datasheet values."""
    # [temp in C, yield strength in psi]
    temp_yield_data = np.array([[-253, 100e3],
                                [-196, 70e3],
                                [-79, 50e3],
                                [21, 35e3],
                                [205, 23e3],
                                [427, 19e3],
                                [650, 15.5e3],
                                [815, 13e3]])
    temps = temp_yield_data[:, 0]
    yields = temp_yield_data[:, 1]

    # np.interp clamps to the end values outside the data, which would overestimate strength above 815 C
    if np.any(np.asarray(T) < temps[0]) or np.any(np.asarray(T) > temps[-1]):
        raise ValueError(f"T = {T} C is outside the data range ({temps[0]:.0f} to {temps[-1]:.0f} C)")

    return np.interp(T, temps, yields)